# Generating NESTML neuron code for SpiNNaker

In this notebook, we will take a NESTML neuron model (the ``iaf_psc_exp`` model), and generate and compile code that will run on SpiNNaker.

First, we need to download and install the SpiNNaker software necessary for new model development.

In [1]:
%cd /opt/app-root/src
!git clone https://github.com/SpiNNakerManchester/SpiNNFrontEndCommon.git --depth=1
!git clone https://github.com/SpiNNakerManchester/sPyNNaker.git --depth=1
!git clone https://github.com/SpiNNakerManchester/spinn_common.git --depth=1
!git clone https://github.com/SpiNNakerManchester/SpiNNUtils --depth=1

Cloning into 'SpiNNFrontEndCommon'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (348/348), done.
remote: Total 400 (delta 99), reused 171 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (400/400), 652.83 KiB | 1.64 MiB/s, done.
Resolving deltas: 100% (99/99), done.
Cloning into 'sPyNNaker'...
remote: Enumerating objects: 1213, done.
remote: Counting objects: 100% (1213/1213), done.
remote: Compressing objects: 100% (981/981), done.
remote: Total 1213 (delta 414), reused 354 (delta 144), pack-reused 0 (from 0)
Receiving objects: 100% (1213/1213), 7.64 MiB | 10.90 MiB/s, done.
Resolving deltas: 100% (414/414), done.
Cloning into 'spinn_common'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 62 (delta 14), reused 17 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 114.71 KiB | 7.6

SpiNNaker is based on 32-bit ARM CPUs. We thus need a C cross-compiler for the ARM platform.

In [2]:
!wget https://developer.arm.com/-/media/Files/downloads/gnu-rm/10.3-2021.10/gcc-arm-none-eabi-10.3-2021.10-x86_64-linux.tar.bz2

--2026-06-10 16:23:55--  https://developer.arm.com/-/media/Files/downloads/gnu-rm/10.3-2021.10/gcc-arm-none-eabi-10.3-2021.10-x86_64-linux.tar.bz2
Resolving developer.arm.com... 92.123.12.146, 92.123.12.152, 2a02:26f0:480:3a::215:41d0, ...
Connecting to developer.arm.com|92.123.12.146|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://armkeil.blob.core.windows.net/developer/Files/downloads/gnu-rm/10.3-2021.10/gcc-arm-none-eabi-10.3-2021.10-x86_64-linux.tar.bz2 [following]
--2026-06-10 16:23:55--  https://armkeil.blob.core.windows.net/developer/Files/downloads/gnu-rm/10.3-2021.10/gcc-arm-none-eabi-10.3-2021.10-x86_64-linux.tar.bz2
Resolving armkeil.blob.core.windows.net... 20.209.58.235
Connecting to armkeil.blob.core.windows.net|20.209.58.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 157089706 (150M) [application/octet-stream]
Saving to: ‘gcc-arm-none-eabi-10.3-2021.10-x86_64-linux.tar.bz2’

gcc-arm-none-ea

In [3]:
!bunzip2 gcc-arm-none-eabi-10.3-2021.10-x86_64-linux.tar.bz2

In [4]:
!tar -xf gcc-arm-none-eabi-10.3-2021.10-x86_64-linux.tar

Now, we can build SpiNNaker-tools:

In [5]:
!git clone https://github.com/SpiNNakerManchester/spinnaker_tools.git
!cd /opt/app-root/src/spinnaker_tools && PATH=$PATH:/opt/app-root/src/gcc-arm-none-eabi-10.3-2021.10/bin make

Cloning into 'spinnaker_tools'...
remote: Enumerating objects: 21930, done.
remote: Counting objects: 100% (888/888), done.
remote: Compressing objects: 100% (435/435), done.
remote: Total 21930 (delta 726), reused 487 (delta 453), pack-reused 21042 (from 3)
Receiving objects: 100% (21930/21930), 16.70 MiB | 14.61 MiB/s, done.
Resolving deltas: 100% (19317/19317), done.
make[1]: Entering directory '/opt/app-root/src/spinnaker_tools/sark'
mkdir -p /opt/app-root/src/spinnaker_tools/build/gnu
/opt/app-root/src/spinnaker_tools/tools/arm2gas sark_alib.s > /opt/app-root/src/spinnaker_tools/build/gnu/sark_alib.gas
mkdir -p /opt/app-root/src/spinnaker_tools/build/gnu
/opt/app-root/src/spinnaker_tools/tools/h2asm /opt/app-root/src/spinnaker_tools/include/spinnaker.h | /opt/app-root/src/spinnaker_tools/tools/arm2gas > /opt/app-root/src/spinnaker_tools/build/gnu/spinnaker.gas
mkdir -p /opt/app-root/src/spinnaker_tools/build/gnu
/opt/app-root/src/spinnaker_tools/tools/h2asm /opt/app-root/src/spinn

Then build spinn_common:

In [6]:
!cd /opt/app-root/src/spinn_common && CPATH=$CPATH:/opt/app-root/src/spinnaker-install/c_models/modified_src:/opt/app-root/src/spinn_common/include PATH=$PATH:/opt/app-root/src/gcc-arm-none-eabi-10.3-2021.10/bin PYTHONPATH=$PYTHONPATH:/opt/app-root/src/SpiNNUtils make

# this builds ./lib/libspinn_common.a

make -f spinn_common_lib.mk all
make[1]: Entering directory '/opt/app-root/src/spinn_common'
arm-none-eabi-gcc -c -mthumb-interwork -march=armv5te -std=gnu99 -ffreestanding -I /opt/app-root/src/spinnaker_tools/include -fdata-sections -ffunction-sections -g -I /opt/app-root/src/spinn_common//include -Ofast -DPRODUCTION_CODE -o /opt/app-root/src/spinn_common//build/bit_field.o src/bit_field.c
arm-none-eabi-gcc -c -mthumb-interwork -march=armv5te -std=gnu99 -ffreestanding -I /opt/app-root/src/spinnaker_tools/include -fdata-sections -ffunction-sections -g -I /opt/app-root/src/spinn_common//include -Ofast -DPRODUCTION_CODE -o /opt/app-root/src/spinn_common//build/circular_buffer.o src/circular_buffer.c
arm-none-eabi-gcc -c -mthumb-interwork -march=armv5te -std=gnu99 -ffreestanding -I /opt/app-root/src/spinnaker_tools/include -fdata-sections -ffunction-sections -g -I /opt/app-root/src/spinn_common//include -Ofast -DPRODUCTION_CODE -o /opt/app-root/src/spinn_common//build/normal.o src/normal.

Then build SpiNNFrontEndCommon:

In [7]:
!cd /opt/app-root/src/SpiNNFrontEndCommon/c_common && CPATH=$CPATH:/opt/app-root/src/spinnaker-install/c_models/modified_src:/opt/app-root/src/spinn_common/include PATH=$PATH:/opt/app-root/src/gcc-arm-none-eabi-10.3-2021.10/bin PYTHONPATH=/opt/app-root/src/SpiNNUtils:$PYTHONPATH make

make[1]: Entering directory '/opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib'
python -m spinn_utilities.make_tools.converter /opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib//src/ /opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib//build/modified_src/ /opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib/../../spinn_front_end_common/common_model_binaries/ C
arm-none-eabi-gcc -c -mthumb-interwork -march=armv5te -std=gnu99 -ffreestanding -I /opt/app-root/src/spinnaker_tools/include -I /opt/app-root/src/spinn_common/include -fdata-sections -ffunction-sections -g -Os -I /opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib//include -DPRODUCTION_CODE -o /opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib//build/data_specification.o /opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib//build/modified_src//data_specification.c
arm-none-eabi-gcc -c -mthumb-interwork -march=armv5te

Then build neural_modelling:

In [8]:
!cd /opt/app-root/src/sPyNNaker/neural_modelling && CPATH=$CPATH:/opt/app-root/src/spinnaker-install/c_models/modified_src:/opt/app-root/src/spinn_common/include PATH=$PATH:/opt/app-root/src/gcc-arm-none-eabi-10.3-2021.10/bin PYTHONPATH=/opt/app-root/src/SpiNNUtils:$PYTHONPATH make

for d in makefiles/synapse_expander makefiles/spike_source/poisson makefiles/delay_extension makefiles/robot_motor_control makefiles/neuron_only makefiles/synapse_only makefiles/neuron makefiles/local_only_combined; do make -C $d || exit $?; done
make[1]: Entering directory '/opt/app-root/src/sPyNNaker/neural_modelling/makefiles/synapse_expander'
for d in synapse_expander neuron_expander; do make -f $d.mk || exit $?; done
make[2]: Entering directory '/opt/app-root/src/sPyNNaker/neural_modelling/makefiles/synapse_expander'
python -m spinn_utilities.make_tools.converter /opt/app-root/src/sPyNNaker/neural_modelling/src/ /opt/app-root/src/sPyNNaker/neural_modelling/modified_src/ /opt/app-root/src/sPyNNaker/spynnaker/pyNN/model_binaries/ S
# local
arm-none-eabi-gcc -c -mthumb-interwork -march=armv5te -std=gnu99 -ffreestanding -I /opt/app-root/src/spinnaker_tools/include -I /opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib/include -Wall -Wextra -Wold-style-definition -DPROD

## Generating code for the NESTML model

Let's begin by downloading the NESTML model. In this tutorial, we will use the ``iaf_psc_exp`` neuron model.

In [9]:
!wget https://raw.githubusercontent.com/nest/nestml/refs/heads/main/models/neurons/iaf_psc_exp_neuron.nestml

--2026-06-10 16:28:53--  https://raw.githubusercontent.com/nest/nestml/refs/heads/main/models/neurons/iaf_psc_exp_neuron.nestml
Resolving raw.githubusercontent.com... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4501 (4.4K) [text/plain]
Saving to: ‘iaf_psc_exp_neuron.nestml’

iaf_psc_exp_neuron. 100%[===================>]   4.40K  --.-KB/s    in 0s      

2026-06-10 16:28:53 (89.0 MB/s) - ‘iaf_psc_exp_neuron.nestml’ saved [4501/4501]



In [10]:
# clean up the code generation target directories in case they already exist from a previous run
!mv -v spinnaker-target spinnaker-target-$(date +%Y-%m-%d_%H-%M-%S.%N | cut -b1-23)
!mv -v spinnaker-install spinnaker-install-$(date +%Y-%m-%d_%H-%M-%S.%N | cut -b1-23)

# need to create this directory first, otherwise it gets ignored in the PYTHONPATH!
!mkdir spinnaker-install

mv: cannot stat 'spinnaker-target': No such file or directory
mv: cannot stat 'spinnaker-install': No such file or directory


Now, generate the actual code. There will be an error during compilation because the installation steps are a bit different on the EBRAINS Lab. However, no errors should occur in the model itself.

In [11]:
from pynestml.frontend.pynestml_frontend import generate_spinnaker_target

input_path = ["iaf_psc_exp_neuron.nestml"]
target_path = "spinnaker-target"
install_path = "spinnaker-install"
logging_level = "WARNING"   # try "DEBUG" for extra debug information
suffix = "_nestml"
generate_spinnaker_target(input_path,
                          target_path=target_path,
                          install_path=install_path,
                          logging_level=logging_level,
                          suffix=suffix)

Copying "iaf_psc_exp_neuron_nestml_impl.h" to /opt/app-root/src/spinnaker-install/c_models/src/my_models/implementations
'iaf_psc_exp_neuron_nestml_impl.h' -> '/opt/app-root/src/spinnaker-install/c_models/src/my_models/implementations/iaf_psc_exp_neuron_nestml_impl.h'
'__init__.py' -> '/opt/app-root/src/spinnaker-install/python_models8/__init__.py'
Copying "__init__.py" to /opt/app-root/src/spinnaker-install/python_models8/neuron/builds
'__init__.py' -> '/opt/app-root/src/spinnaker-install/python_models8/neuron/builds/__init__.py'
Copying "iaf_psc_exp_neuron_nestml.py" to /opt/app-root/src/spinnaker-install/python_models8/neuron/builds
'iaf_psc_exp_neuron_nestml.py' -> '/opt/app-root/src/spinnaker-install/python_models8/neuron/builds/iaf_psc_exp_neuron_nestml.py'
Copying "iaf_psc_exp_neuron_nestml_impl.py" to /opt/app-root/src/spinnaker-install/python_models8/neuron/implementations
'/opt/app-root/src/spinnaker-target/iaf_psc_exp_neuron_nestml_impl.py' -> '/opt/app-root/src/spinnaker-in

GeneratedCodeBuildException: Error occurred during 'make'! More detailed error messages can be found in stdout.

Now, we compile the generated code into a binary format that has the ".aplx" extension.

In [12]:
# XXX: some commands don't run from Jupyter but work fine when run from the terminal!
# XXX: use a workaround where the commands are written to a temporary file and executed

import os

def write_and_run_script(compilation_script, debug=True):
    """A helper function to write a shellscript to a temporary file and execute it."""
    import tempfile, os, subprocess, stat
    with tempfile.NamedTemporaryFile(mode="w", suffix=".sh", delete=False) as tmp_file:
        tmp_file.write(compilation_script)
        tmp_path = tmp_file.name
    
    os.chmod(tmp_path, stat.S_IRWXU)   # mode 700
    
    result = subprocess.run([tmp_path], capture_output=True, text=True)

    if debug:
        print("=== STDOUT ===")
        print(result.stdout)
        if result.stderr:
            print("=== STDERR ===")
            print(result.stderr)
        print(f"Exit code: {result.returncode}")


In [13]:
# patch debug.h

compilation_script = """#!/bin/bash

cd  /opt/app-root/src

TMPPATCH=$(mktemp /tmp/patch.XXXXXX)
cat >"$TMPPATCH" <<'PATCH_EOF'
--- ./SpiNNFrontEndCommon/c_common/front_end_common_lib/include/debug_orig.h    2026-06-10 14:14:43.747283329 +0200
+++ ./SpiNNFrontEndCommon/c_common/front_end_common_lib/include/debug.h 2026-06-10 14:14:55.107125347 +0200
@@ -81,7 +81,7 @@
 //! handled more efficiently by the binary deployed to SpiNNaker.
 //!
 //! \param[in] message: The user-defined part of the error message.
-extern void log_error(const char *message, ...);
+static void log_error(const char *message, ...) {}
 
 //! \brief This function logs warnings.
 //!
@@ -90,7 +90,7 @@
 //! handled more efficiently by the binary deployed to SpiNNaker.
 //!
 //! \param[in] message: The user-defined part of the error message.
-extern void log_warning(const char *message, ...);
+static void log_warning(const char *message, ...) {}
 
 //! \brief This function logs informational messages. This is the lowest level
 //!        of message normally printed.
@@ -100,7 +100,7 @@
 //! handled more efficiently by the binary deployed to SpiNNaker.
 //!
 //! \param[in] message: The user-defined part of the error message.
-extern void log_info(const char *message, ...);
+static void log_info(const char *message, ...) {}
 
 //! \brief This function logs debugging messages. This level of message is
 //! normally not printed except when the binary is built in debug mode
@@ -110,7 +110,7 @@
 //! handled more efficiently by the binary deployed to SpiNNaker.
 //!
 //! \param[in] message: The user-defined part of the error message.
-extern void log_debug(const char *message, ...);
+static void log_debug(const char *message, ...) {}
 
 //! \brief Type-pun a float as a 32-bit unsigned integer.
 //!
PATCH_EOF

cat "$TMPPATCH"

patch -p1 < "$TMPPATCH"
"""

write_and_run_script(compilation_script)

=== STDOUT ===
--- ./SpiNNFrontEndCommon/c_common/front_end_common_lib/include/debug_orig.h    2026-06-10 14:14:43.747283329 +0200
+++ ./SpiNNFrontEndCommon/c_common/front_end_common_lib/include/debug.h 2026-06-10 14:14:55.107125347 +0200
@@ -81,7 +81,7 @@
 //! handled more efficiently by the binary deployed to SpiNNaker.
 //!
 //! \param[in] message: The user-defined part of the error message.
-extern void log_error(const char *message, ...);
+static void log_error(const char *message, ...) {}
 
 //!rief This function logs warnings.
 //!
@@ -90,7 +90,7 @@
 //! handled more efficiently by the binary deployed to SpiNNaker.
 //!
 //! \param[in] message: The user-defined part of the error message.
-extern void log_warning(const char *message, ...);
+static void log_warning(const char *message, ...) {}
 
 //!rief This function logs informational messages. This is the lowest level
 //!        of message normally printed.
@@ -100,7 +100,7 @@
 //! handled more efficiently by the binary deploy

In [14]:
compilation_script = """#!/bin/bash

cd  /opt/app-root/src/SpiNNFrontEndCommon

TMPPATCH=$(mktemp /tmp/patch.XXXXXX)
cat >"$TMPPATCH" <<'PATCH_EOF'
--- ./c_common/front_end_common_lib/make/fec.mk 2026-06-09 22:09:22.952143369 +0200
+++ ./c_common/front_end_common_lib/make/fec.mk 2026-06-10 15:52:59.459116627 +0200
@@ -70,7 +70,7 @@
 # directory pair
 define add_source_dir#(src_dir)
 $(call get_path,$(1),2): $(wildcard $(call get_path,$(1),1)/**/*)
-	python -m spinn_utilities.make_tools.converter $(call get_path,$(1),1) $(call get_path,$(1),2) $(APP_OUTPUT_DIR) $(DATABASE_KEY)
+	echo ">>>>>>>> python -m spinn_utilities.make_tools.converter $(call get_path,$(1),1) $(call get_path,$(1),2) $(APP_OUTPUT_DIR) $(DATABASE_KEY)"
 
 $(call get_path,$(1), 2)%.c: $(call get_path,$(1), 1)%.c
 	python -m spinn_utilities.make_tools.converter $(call get_path,$(1),1) $(call get_path,$(1),2) $(APP_OUTPUT_DIR) $(DATABASE_KEY)
PATCH_EOF

cat "$TMPPATCH"

patch -p0 < "$TMPPATCH"
"""

write_and_run_script(compilation_script)

=== STDOUT ===
--- ./c_common/front_end_common_lib/make/fec.mk 2026-06-09 22:09:22.952143369 +0200
+++ ./c_common/front_end_common_lib/make/fec.mk 2026-06-10 15:52:59.459116627 +0200
@@ -70,7 +70,7 @@
 # directory pair
 define add_source_dir#(src_dir)
 $(call get_path,$(1),2): $(wildcard $(call get_path,$(1),1)/**/*)
-	python -m spinn_utilities.make_tools.converter $(call get_path,$(1),1) $(call get_path,$(1),2) $(APP_OUTPUT_DIR) $(DATABASE_KEY)
+	echo ">>>>>>>> python -m spinn_utilities.make_tools.converter $(call get_path,$(1),1) $(call get_path,$(1),2) $(APP_OUTPUT_DIR) $(DATABASE_KEY)"
 
 $(call get_path,$(1), 2)%.c: $(call get_path,$(1), 1)%.c
 	python -m spinn_utilities.make_tools.converter $(call get_path,$(1),1) $(call get_path,$(1),2) $(APP_OUTPUT_DIR) $(DATABASE_KEY)
patching file ./c_common/front_end_common_lib/make/fec.mk

Exit code: 0


In [15]:
# in /opt/app-root/src/spinnaker-install/c_models/makefiles/iaf_psc_exp_neuron_nestml_impl/Makefile, 
#      replace "$(EXTRA_SRC_DIR)/...."
#      by "/opt/app-root/src/spinnaker-install/c_models/modified_src/my_models/implementations/iaf_psc_exp_neuron_nestml_impl.h"

!sed -i 's/\$(EXTRA_SRC_DIR)/\/opt\/app-root\/src\/spinnaker-install\/c_models\/modified_src/g' /opt/app-root/src/spinnaker-install/c_models/makefiles/iaf_psc_exp_neuron_nestml_impl/Makefile

In [19]:
# fix issue with empty -include 

compilation_script = """#!/bin/bash

cd /opt/app-root/src/sPyNNaker

TMPPATCH=$(mktemp /tmp/patch.XXXXXX)
cat >"$TMPPATCH" <<'PATCH_EOF'
--- neural_modelling/make/neural_build.mk      2026-06-10 16:09:46.768799882 +0200
+++ neural_modelling/make/neural_build.mk   2026-06-10 16:10:34.072131869 +0200
@@ -299,9 +299,9 @@
 $(BUILD_DIR)neuron/neuron.o: $(NEURON_MODIFIED_DIR)neuron/neuron.c $(NEURON_INCLUDE_FILES) $(MAKEFILE_LIST)
 	# neuron.o
 	-@mkdir -p $(dir $@)
-	$(CC) -DLOG_LEVEL=$(NEURON_DEBUG) $(CFLAGS) $(NEURON_INCLUDES) -o $@ $<
+	$(CC) -DLOG_LEVEL=$(NEURON_DEBUG) $(CFLAGS) -include /opt/app-root/src/spinnaker-install/c_models/src/my_models/implementations/iaf_psc_exp_neuron_nestml_impl.h -o $@ $<
 
 .PRECIOUS: $(NEURON_MODIFIED_DIR)%.c $(NEURON_MODIFIED_DIR)%.h $(LOG_DICT_FILE) $(EXTRA_PRECIOUS)
PATCH_EOF

cat "$TMPPATCH"

patch -p0 < "$TMPPATCH"
"""

write_and_run_script(compilation_script)

=== STDOUT ===
--- neural_modelling/make/neural_build.mk      2026-06-10 16:09:46.768799882 +0200
+++ neural_modelling/make/neural_build.mk   2026-06-10 16:10:34.072131869 +0200
@@ -299,9 +299,9 @@
 $(BUILD_DIR)neuron/neuron.o: $(NEURON_MODIFIED_DIR)neuron/neuron.c $(NEURON_INCLUDE_FILES) $(MAKEFILE_LIST)
 	# neuron.o
 	-@mkdir -p $(dir $@)
-	$(CC) -DLOG_LEVEL=$(NEURON_DEBUG) $(CFLAGS) $(NEURON_INCLUDES) -o $@ $<
+	$(CC) -DLOG_LEVEL=$(NEURON_DEBUG) $(CFLAGS) -include /opt/app-root/src/spinnaker-install/c_models/src/my_models/implementations/iaf_psc_exp_neuron_nestml_impl.h -o $@ $<
 
 .PRECIOUS: $(NEURON_MODIFIED_DIR)%.c $(NEURON_MODIFIED_DIR)%.h $(LOG_DICT_FILE) $(EXTRA_PRECIOUS)
patching file neural_modelling/make/neural_build.mk
Hunk #1 succeeded at 299 with fuzz 2.

Exit code: 0


In [20]:
# first build -- this will fail
!cd /opt/app-root/src/spinnaker-install/c_models/ && source /opt/app-root/src/sPyNNaker/neural_modelling/setup.bash && FEC_INSTALL_DIR=/opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib SPYNNAKER_INSTALL_DIR=/opt/app-root/src/sPyNNaker CPATH=$CPATH:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/current_sources:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/additional_inputs:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/threshold_types:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/input_types:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/synapse_types:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/models:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron:/opt/app-root/src/spinnaker-install/c_models/modified_src:/opt/app-root/src/spinn_common/include PATH=$PATH:/opt/app-root/src/gcc-arm-none-eabi-10.3-2021.10/bin PYTHONPATH=$PYTHONPATH:/opt/app-root/src/SpiNNUtils make

# for some reason, neuron.o is not generated for the NESTML model. Copy it from somewhere else
!cp -v /opt/app-root/src/sPyNNaker/neural_modelling/builds/IF_curr_alpha_stdp_mad_pair_additive/neuron/neuron.o /opt/app-root/src/sPyNNaker/neural_modelling/builds/iaf_psc_exp_neuron_nestml_impl/neuron

# copy unmodified (!) src to modified_src
!cp -rv /opt/app-root/src/sPyNNaker/neural_modelling/src/neuron /opt/app-root/src/sPyNNaker/neural_modelling/modified_src/

# do the actual compilation
!cd /opt/app-root/src/spinnaker-install/c_models/ && source /opt/app-root/src/sPyNNaker/neural_modelling/setup.bash && FEC_INSTALL_DIR=/opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib SPYNNAKER_INSTALL_DIR=/opt/app-root/src/sPyNNaker CPATH=$CPATH:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/current_sources:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/additional_inputs:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/threshold_types:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/input_types:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/synapse_types:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron/models:/opt/app-root/src/sPyNNaker/neural_modelling/src/neuron:/opt/app-root/src/spinnaker-install/c_models/modified_src:/opt/app-root/src/spinn_common/include PATH=$PATH:/opt/app-root/src/gcc-arm-none-eabi-10.3-2021.10/bin PYTHONPATH=$PYTHONPATH:/opt/app-root/src/SpiNNUtils make

for d in makefiles; do make -C $d || exit $?; done
make[1]: Entering directory '/opt/app-root/src/spinnaker-install/c_models/makefiles'
for d in iaf_psc_exp_neuron_nestml_impl ; do make -C $d || exit $?; done
make[2]: Entering directory '/opt/app-root/src/spinnaker-install/c_models/makefiles/iaf_psc_exp_neuron_nestml_impl'
echo ">>>>>>>> python -m spinn_utilities.make_tools.converter /opt/app-root/src/spinnaker-install/c_models/src/ / /opt/app-root/src/sPyNNaker/spynnaker/pyNN/model_binaries/ S"
>>>>>>>> python -m spinn_utilities.make_tools.converter /opt/app-root/src/spinnaker-install/c_models/src/ / /opt/app-root/src/sPyNNaker/spynnaker/pyNN/model_binaries/ S
#c_main.c
arm-none-eabi-gcc -c -mthumb-interwork -march=armv5te -std=gnu99 -ffreestanding -I /opt/app-root/src/spinnaker_tools/include -DLOG_LEVEL=LOG_INFO -I/opt/app-root/src/spinnaker-install/c_models/src -I /opt/app-root/src/SpiNNFrontEndCommon/c_common/front_end_common_lib/include -Wall -Wextra -Wold-style-definition -DPRODU

In [21]:
# clean up the generated sPyNNaker wrapper

!sed -i 's/from python_models8.neuron.implementations.iaf_psc_exp_neuron_nestml_impl import iaf_psc_exp_neuron_nestmlImpl/from iaf_psc_exp_neuron_nestml_impl import iaf_psc_exp_neuron_nestmlImpl/g' /opt/app-root/src/spinnaker-install/python_models8/neuron/builds/iaf_psc_exp_neuron_nestml.py

Now, the aplx file should have been generated and is ready for use on SpiNNaker!

For convenience, we copy it to the home directory.

In [25]:
!cp -v /opt/app-root/src/sPyNNaker/spynnaker/pyNN/model_binaries/iaf_psc_exp_neuron_nestml_impl.aplx ~

'/opt/app-root/src/sPyNNaker/spynnaker/pyNN/model_binaries/iaf_psc_exp_neuron_nestml_impl.aplx' -> '/opt/app-root/src/iaf_psc_exp_neuron_nestml_impl.aplx'


Note that two Python wrapper scripts were also generated. These are needed to be able to instantiate and use the model via the sPyNNaker Python interface of SpiNNaker.

In [26]:
!cp -v /opt/app-root/src/spinnaker-install/python_models8/neuron/implementations/iaf_psc_exp_neuron_nestml_impl.py ~
!cp -v /opt/app-root/src/spinnaker-install/python_models8/neuron/builds/iaf_psc_exp_neuron_nestml.py ~

'/opt/app-root/src/spinnaker-install/python_models8/neuron/implementations/iaf_psc_exp_neuron_nestml_impl.py' -> '/opt/app-root/src/iaf_psc_exp_neuron_nestml_impl.py'
'/opt/app-root/src/spinnaker-install/python_models8/neuron/builds/iaf_psc_exp_neuron_nestml.py' -> '/opt/app-root/src/iaf_psc_exp_neuron_nestml.py'


## Acknowledgements

The authors would like to thank Andrew Rowley for his kind and helpful discussion and feedback.

## Copyright

This file is part of NEST.

Copyright (C) 2004 The NEST Initiative

NEST is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 2 of the License, or (at your option) any later version.

NEST is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with NEST. If not, see http://www.gnu.org/licenses/.